# SimData v4

Session-centric notebook aligned with the new generator:

- Isolation Forest + XGBoost for binary anomaly detection
- Random Forest for anomaly type classification
- Markov chain for next-action prediction
- Prophet for daily trend forecasting
- KMeans clustering, path deviation mining, churn prediction, and risk scoring

In [ ]:
# !pip install -q pandas numpy scikit-learn matplotlib seaborn xgboost prophet joblib

In [ ]:
import json, math, os, warnings, joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    from prophet import Prophet
except Exception:
    Prophet = None

try:
    from google.colab import files
except Exception:
    files = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
ARTIFACTS_DIR = Path("artifacts_v4_tabular")
ARTIFACTS_DIR.mkdir(exist_ok=True)
DOWNLOAD_WORDS = ("download", "document", "card", "wallet", "certificate", "refund")

In [ ]:
UPLOAD = False
if UPLOAD and files is not None:
    files.upload()

EVENT_CSV = os.environ.get("SIMDATA_EVENT_CSV", "audit_trail_2025.csv")
SESSION_CSV = os.environ.get(
    "SIMDATA_SESSION_CSV",
    str(Path(EVENT_CSV).with_name(f"{Path(EVENT_CSV).stem}_session_summary{Path(EVENT_CSV).suffix or '.csv'}")),
)


def maybe_read_csv(path: str):
    return pd.read_csv(path) if path and Path(path).exists() else None


def longest_ko_streak(values):
    longest = current = 0
    for value in values:
        if value == "KO":
            current += 1
            longest = max(longest, current)
        else:
            current = 0
    return int(longest)


def is_download_action(value: str) -> int:
    value = str(value).lower()
    return int(any(word in value for word in DOWNLOAD_WORDS))


def normalize_events(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["createdAt"] = pd.to_datetime(df["createdAt"], utc=True, errors="coerce")
    df = df.dropna(subset=["createdAt"]).copy()
    fill_text = {
        "action": "UNKNOWN_ACTION",
        "type": "UNKNOWN_TYPE",
        "subType": "NONE",
        "device": "UNKNOWN_DEVICE",
        "countryCode": "UNKNOWN_COUNTRY",
        "city": "UNKNOWN_CITY",
        "status": "UNKNOWN_STATUS",
        "persona": "unknown",
        "route": "unknown",
        "anomaly_type": "normal",
    }
    for col, default in fill_text.items():
        if col not in df.columns:
            df[col] = default
        df[col] = df[col].fillna(default).astype(str)
    if "sessionId" not in df.columns:
        df["sessionId"] = df["insuredId"].astype(str) + "-" + df["sessionNumber"].astype(str)
    if "sessionNumber" not in df.columns:
        df["sessionNumber"] = 0
    if "sequenceInSession" not in df.columns:
        df["sequenceInSession"] = df.groupby("sessionId").cumcount() + 1
    if "sessionLength" not in df.columns:
        df["sessionLength"] = df.groupby("sessionId")["sequenceInSession"].transform("max")
    if "is_anomaly" not in df.columns:
        df["is_anomaly"] = 0
    df["is_anomaly"] = pd.to_numeric(df["is_anomaly"], errors="coerce").fillna(0).astype(int)
    df = df.sort_values(["sessionId", "sequenceInSession", "createdAt"]).reset_index(drop=True)
    g = df.groupby("sessionId", sort=False)
    df["prevAction"] = g["action"].shift(1).fillna("")
    df["nextAction"] = g["action"].shift(-1).fillna("")
    delta = (df["createdAt"] - g["createdAt"].shift(1)).dt.total_seconds().fillna(0)
    df["timeDeltaSinceLastAction"] = pd.to_numeric(df.get("timeDeltaSinceLastAction", np.nan), errors="coerce").fillna(delta)
    df["hourOfDay"] = pd.to_numeric(df.get("hourOfDay", np.nan), errors="coerce").fillna(df["createdAt"].dt.hour)
    df["dayOfWeek"] = pd.to_numeric(df.get("dayOfWeek", np.nan), errors="coerce").fillna(df["createdAt"].dt.dayofweek)
    df["isWeekend"] = pd.to_numeric(df.get("isWeekend", np.nan), errors="coerce").fillna((df["dayOfWeek"] >= 5).astype(int))
    df["isDownloadAction"] = pd.to_numeric(df.get("isDownloadAction", np.nan), errors="coerce").fillna(df["action"].map(is_download_action))
    df["cumulativeKOs"] = pd.to_numeric(df.get("cumulativeKOs", np.nan), errors="coerce").fillna(g["status"].transform(lambda s: (s == "KO").cumsum()))
    df["hasLoggedIn"] = pd.to_numeric(df.get("hasLoggedIn", np.nan), errors="coerce").fillna(
        g["action"].transform(lambda s: s.isin(["Connexion", "Connexion SSO", "Connexion en tant que"]).cumsum().shift(fill_value=0))
    )
    if "ip" in df.columns:
        first_ip = g["ip"].transform("first")
        df["isIpChanged"] = pd.to_numeric(df.get("isIpChanged", np.nan), errors="coerce").fillna((df["ip"] != first_ip).astype(int))
        df["uniqueIpsInSession"] = pd.to_numeric(df.get("uniqueIpsInSession", np.nan), errors="coerce").fillna(g["ip"].transform("nunique"))
    else:
        df["isIpChanged"] = 0
        df["uniqueIpsInSession"] = 1
    first_device = g["device"].transform("first")
    df["isDeviceChanged"] = pd.to_numeric(df.get("isDeviceChanged", np.nan), errors="coerce").fillna((df["device"] != first_device).astype(int))
    df["uniqueDevicesInSession"] = pd.to_numeric(df.get("uniqueDevicesInSession", np.nan), errors="coerce").fillna(g["device"].transform("nunique"))
    df["downloadActionsInSession"] = pd.to_numeric(df.get("downloadActionsInSession", np.nan), errors="coerce").fillna(g["isDownloadAction"].cumsum())
    df["sessionDurationSeconds"] = pd.to_numeric(df.get("sessionDurationSeconds", np.nan), errors="coerce").fillna(
        g["createdAt"].transform(lambda s: (s.max() - s.min()).total_seconds())
    )
    df["pingPongCount"] = pd.to_numeric(df.get("pingPongCount", np.nan), errors="coerce").fillna(0)
    df["downloadsLast2Minutes"] = pd.to_numeric(df.get("downloadsLast2Minutes", np.nan), errors="coerce").fillna(df["downloadActionsInSession"])
    df["eventDate"] = df["createdAt"].dt.floor("D")
    return df


def derive_sessions(events: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for session_id, g in events.groupby("sessionId", sort=False):
        g = g.sort_values(["sequenceInSession", "createdAt"]).reset_index(drop=True)
        start_ts = g["createdAt"].iloc[0]
        end_ts = g["createdAt"].iloc[-1]
        anomaly_values = [value for value in g["anomaly_type"].tolist() if value != "normal"]
        counts = pd.Series(anomaly_values).value_counts()
        rows.append({
            "sessionId": session_id,
            "insuredId": str(g["insuredId"].iloc[0]),
            "persona": g["persona"].iloc[0],
            "countryCode": g["countryCode"].iloc[0],
            "city": g["city"].iloc[0],
            "sessionNumber": int(g["sessionNumber"].iloc[0]),
            "sessionStart": start_ts,
            "sessionEnd": end_ts,
            "startHour": int(start_ts.hour),
            "endHour": int(end_ts.hour),
            "dayOfWeek": int(start_ts.dayofweek),
            "isWeekend": int(start_ts.dayofweek >= 5),
            "firstAction": g["action"].iloc[0],
            "lastAction": g["action"].iloc[-1],
            "firstRoute": g["route"].iloc[0],
            "lastRoute": g["route"].iloc[-1],
            "totalEvents": int(len(g)),
            "totalDurationSeconds": float((end_ts - start_ts).total_seconds()),
            "avgInterActionSeconds": float(g["timeDeltaSinceLastAction"].iloc[1:].mean() if len(g) > 1 else 0),
            "minInterActionSeconds": float(g["timeDeltaSinceLastAction"].iloc[1:].min() if len(g) > 1 else 0),
            "maxInterActionSeconds": float(g["timeDeltaSinceLastAction"].iloc[1:].max() if len(g) > 1 else 0),
            "uniqueActions": int(g["action"].nunique()),
            "uniqueRoutes": int(g["route"].nunique()),
            "uniqueIpsUsed": int(g["uniqueIpsInSession"].max()),
            "uniqueDevicesUsed": int(g["uniqueDevicesInSession"].max()),
            "totalKOs": int((g["status"] == "KO").sum()),
            "totalOKs": int((g["status"] == "OK").sum()),
            "longestKoStreak": int(longest_ko_streak(g["status"].tolist())),
            "hasLogin": int(g["action"].isin(["Connexion", "Connexion SSO", "Connexion en tant que"]).any()),
            "hasLogout": int(g["action"].isin(["Déconnexion", "Deconnexion", "SSO Disconnect"]).any()),
            "ipChanged": int(g["isIpChanged"].max()),
            "deviceChanged": int(g["isDeviceChanged"].max()),
            "totalDownloadActions": int(g["isDownloadAction"].sum()),
            "maxDownloadsIn2Minutes": int(g["downloadsLast2Minutes"].max()),
            "pingPongCount": int(g["pingPongCount"].max()),
            "endedAbruptly": int(not g["action"].isin(["Déconnexion", "Deconnexion", "SSO Disconnect"]).any()),
            "is_anomaly": int(g["is_anomaly"].max()),
            "primary_anomaly_type": counts.index[0] if not counts.empty else "normal",
        })
    return pd.DataFrame(rows).sort_values("sessionStart").reset_index(drop=True)


def merge_sessions(raw_sessions, derived_sessions):
    if raw_sessions is None or raw_sessions.empty:
        return derived_sessions
    raw_sessions = raw_sessions.copy()
    raw_sessions["sessionId"] = raw_sessions["sessionId"].astype(str)
    for col in ["sessionStart", "sessionEnd"]:
        if col in raw_sessions.columns:
            raw_sessions[col] = pd.to_datetime(raw_sessions[col], utc=True, errors="coerce")
    merged = derived_sessions.merge(raw_sessions, on="sessionId", how="left", suffixes=("", "_src"))
    for col in raw_sessions.columns:
        if col == "sessionId":
            continue
        src = f"{col}_src"
        if src in merged.columns:
            if col in merged.columns:
                merged[col] = merged[col].where(merged[col].notna(), merged[src])
            else:
                merged[col] = merged[src]
    return merged[[col for col in merged.columns if not col.endswith("_src")]].sort_values("sessionStart").reset_index(drop=True)


def make_matrix(train_df, test_df, num_cols, cat_cols):
    train_df, test_df = train_df.copy(), test_df.copy()
    medians = {}
    for col in num_cols:
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")
        med = float(train_df[col].median()) if train_df[col].notna().any() else 0.0
        medians[col] = med
        train_df[col] = train_df[col].fillna(med)
        test_df[col] = test_df[col].fillna(med)
    for col in cat_cols:
        train_df[col] = train_df[col].fillna("UNKNOWN").astype(str)
        test_df[col] = test_df[col].fillna("UNKNOWN").astype(str)
    train_x = pd.concat([train_df[num_cols].astype(float).reset_index(drop=True), pd.get_dummies(train_df[cat_cols].reset_index(drop=True))], axis=1)
    test_x = pd.concat([test_df[num_cols].astype(float).reset_index(drop=True), pd.get_dummies(test_df[cat_cols].reset_index(drop=True))], axis=1)
    test_x = test_x.reindex(columns=train_x.columns, fill_value=0)
    return train_x, test_x, medians


event_raw = maybe_read_csv(EVENT_CSV)
if event_raw is None:
    raise FileNotFoundError(f"Could not find {EVENT_CSV}")
events = normalize_events(event_raw)
sessions = merge_sessions(maybe_read_csv(SESSION_CSV), derive_sessions(events))

print("events:", events.shape, "sessions:", sessions.shape)
print(events.head(2))
print(sessions.head(2))

In [ ]:
num_cols = [
    "totalEvents", "totalDurationSeconds", "avgInterActionSeconds", "minInterActionSeconds",
    "maxInterActionSeconds", "uniqueActions", "uniqueRoutes", "uniqueIpsUsed",
    "uniqueDevicesUsed", "totalKOs", "totalOKs", "longestKoStreak", "hasLogin",
    "hasLogout", "ipChanged", "deviceChanged", "totalDownloadActions",
    "maxDownloadsIn2Minutes", "pingPongCount", "startHour", "endHour",
    "dayOfWeek", "isWeekend",
]
cat_cols = ["persona", "countryCode", "firstAction", "lastAction", "firstRoute", "lastRoute"]

sessions = sessions.sort_values("sessionStart").reset_index(drop=True)
split = max(1, int(len(sessions) * 0.8))
split = min(split, len(sessions) - 1) if len(sessions) > 1 else 1
train_sessions = sessions.iloc[:split].copy()
test_sessions = sessions.iloc[split:].copy()

X_train, X_test, num_medians = make_matrix(train_sessions, test_sessions, num_cols, cat_cols)
y_train = train_sessions["is_anomaly"].astype(int).values
y_test = test_sessions["is_anomaly"].astype(int).values

print("train sessions:", len(train_sessions), "test sessions:", len(test_sessions))
print("feature matrix:", X_train.shape, X_test.shape)

In [ ]:
contamination = float(train_sessions["is_anomaly"].mean())
contamination = min(max(contamination, 0.01), 0.20)
iso = IsolationForest(n_estimators=400, contamination=contamination, random_state=RANDOM_STATE)
iso.fit(X_train[y_train == 0] if (y_train == 0).any() else X_train)
iso_train_score = -iso.score_samples(X_train[y_train == 0] if (y_train == 0).any() else X_train)
iso_test_score = -iso.score_samples(X_test)
iso_threshold = float(np.quantile(iso_train_score, 1 - contamination))
iso_pred = (iso_test_score >= iso_threshold).astype(int)

print("Isolation Forest threshold:", round(iso_threshold, 4))
print(classification_report(y_test, iso_pred, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(y_test, iso_pred)).plot(colorbar=False)
plt.title("Isolation Forest")
plt.tight_layout()
plt.show()
if len(np.unique(y_test)) > 1:
    print("Isolation Forest ROC-AUC:", round(roc_auc_score(y_test, iso_test_score), 4))

xgb = None
xgb_prob = None
if XGBClassifier is not None:
    pos = max(int(y_train.sum()), 1)
    neg = max(int((y_train == 0).sum()), 1)
    xgb = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        scale_pos_weight=neg / pos,
    )
    xgb.fit(X_train, y_train)
    xgb_prob = xgb.predict_proba(X_test)[:, 1]
    xgb_pred = (xgb_prob >= 0.5).astype(int)
    print(classification_report(y_test, xgb_pred, zero_division=0))
    if len(np.unique(y_test)) > 1:
        print("XGBoost ROC-AUC:", round(roc_auc_score(y_test, xgb_prob), 4))
    imp = pd.DataFrame({"feature": X_train.columns, "importance": xgb.feature_importances_}).sort_values("importance", ascending=False).head(20)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=imp, x="importance", y="feature", orient="h")
    plt.title("Top Features - XGBoost Binary Detector")
    plt.tight_layout()
    plt.show()
    imp.to_csv(ARTIFACTS_DIR / "binary_detector_feature_importance.csv", index=False)
else:
    print("XGBoost not available; binary detector stays unsupervised.")

In [ ]:
anom_train_mask = train_sessions["primary_anomaly_type"] != "normal"
anom_test_mask = test_sessions["primary_anomaly_type"] != "normal"
anom_train = train_sessions[anom_train_mask].copy()
anom_test = test_sessions[anom_test_mask].copy()
type_model = None
type_le = None

if len(anom_train) >= 10 and len(anom_test) > 0 and anom_train["primary_anomaly_type"].nunique() >= 2:
    X_type_train = X_train.loc[anom_train_mask.values]
    X_type_test = X_test.loc[anom_test_mask.values]
    type_le = LabelEncoder()
    y_type_train = type_le.fit_transform(anom_train["primary_anomaly_type"])
    y_type_test = type_le.transform(anom_test["primary_anomaly_type"])
    type_model = RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    type_model.fit(X_type_train, y_type_train)
    type_pred = type_model.predict(X_type_test)
    print("anomaly type classes:", list(type_le.classes_))
    print(classification_report(y_type_test, type_pred, target_names=type_le.classes_, zero_division=0))
    ConfusionMatrixDisplay(confusion_matrix(y_type_test, type_pred), display_labels=type_le.classes_).plot(colorbar=False)
    plt.title("Random Forest - Anomaly Type")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
    imp = pd.DataFrame({"feature": X_train.columns, "importance": type_model.feature_importances_}).sort_values("importance", ascending=False).head(20)
    imp.to_csv(ARTIFACTS_DIR / "anomaly_type_feature_importance.csv", index=False)
else:
    print("Not enough anomalous sessions to train the type classifier.")

In [ ]:
train_ids = set(train_sessions["sessionId"].astype(str))
test_ids = set(test_sessions["sessionId"].astype(str))
train_events = events[events["sessionId"].astype(str).isin(train_ids)].copy()
test_events = events[events["sessionId"].astype(str).isin(test_ids)].copy()


def transition_table(frame: pd.DataFrame) -> pd.DataFrame:
    pairs = []
    for _, g in frame.groupby("sessionId", sort=False):
        g = g.sort_values(["sequenceInSession", "createdAt"])
        actions = g["action"].tolist()
        pairs.extend(list(zip(actions[:-1], actions[1:])))
    out = pd.DataFrame(pairs, columns=["from_action", "to_action"])
    out["count"] = 1
    out = out.groupby(["from_action", "to_action"], as_index=False)["count"].sum().sort_values("count", ascending=False)
    out["probability"] = out.groupby("from_action")["count"].transform(lambda x: x / x.sum())
    return out


transitions = transition_table(train_events)
lookup = {src: grp.sort_values(["probability", "count"], ascending=False) for src, grp in transitions.groupby("from_action", sort=False)}

eval_rows = []
for _, g in test_events.groupby("sessionId", sort=False):
    g = g.sort_values(["sequenceInSession", "createdAt"])
    acts = g["action"].tolist()
    for cur, nxt in zip(acts[:-1], acts[1:]):
        preds = lookup[cur]["to_action"].head(3).tolist() if cur in lookup else []
        eval_rows.append({
            "current_action": cur,
            "actual_next_action": nxt,
            "top1_hit": int(bool(preds) and nxt == preds[0]),
            "top3_hit": int(nxt in preds),
            "train_probability": float(lookup[cur].set_index("to_action")["probability"].get(nxt, 0.0)) if cur in lookup else 0.0,
        })
eval_df = pd.DataFrame(eval_rows)

if not eval_df.empty:
    print("Markov top-1:", round(float(eval_df["top1_hit"].mean()), 4))
    print("Markov top-3:", round(float(eval_df["top3_hit"].mean()), 4))
rare_paths = eval_df[eval_df["train_probability"] < 0.02].copy()
rare_paths.to_csv(ARTIFACTS_DIR / "path_deviations.csv", index=False)
transitions.to_csv(ARTIFACTS_DIR / "markov_transitions.csv", index=False)
print(transitions.head(20))

In [ ]:
cluster_cols = [
    "totalEvents", "totalDurationSeconds", "avgInterActionSeconds", "uniqueActions",
    "totalKOs", "uniqueIpsUsed", "uniqueDevicesUsed", "totalDownloadActions",
    "maxDownloadsIn2Minutes", "pingPongCount",
]
cluster_frame = sessions[cluster_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
cluster_scaler = StandardScaler()
cluster_x = cluster_scaler.fit_transform(cluster_frame)
n_clusters = 4 if len(sessions) >= 40 else max(1, min(3, len(sessions)))
kmeans = KMeans(n_clusters=n_clusters, n_init=20, random_state=RANDOM_STATE)
sessions["cluster_id"] = kmeans.fit_predict(cluster_x)
print(sessions.groupby("cluster_id")[cluster_cols + ["is_anomaly", "endedAbruptly"]].mean().round(2))

pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(cluster_x)
plot_df = pd.DataFrame({"pc1": coords[:, 0], "pc2": coords[:, 1], "cluster_id": sessions["cluster_id"].astype(str), "is_anomaly": sessions["is_anomaly"].astype(int)})
plt.figure(figsize=(8, 6))
sns.scatterplot(data=plot_df, x="pc1", y="pc2", hue="cluster_id", style="is_anomaly", alpha=0.8)
plt.title("Session Persona Clusters")
plt.tight_layout()
plt.show()

churn_num = [col for col in num_cols if col not in {"hasLogout"}]
churn_cat = [col for col in cat_cols if col not in {"lastAction"}]
X_churn_train, X_churn_test, churn_medians = make_matrix(train_sessions, test_sessions, churn_num, churn_cat)
y_churn_train = train_sessions["endedAbruptly"].astype(int).values
y_churn_test = test_sessions["endedAbruptly"].astype(int).values
churn_model = RandomForestClassifier(n_estimators=500, min_samples_leaf=2, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1)
churn_model.fit(X_churn_train, y_churn_train)
churn_prob = churn_model.predict_proba(X_churn_test)[:, 1]
churn_pred = (churn_prob >= 0.5).astype(int)
print(classification_report(y_churn_test, churn_pred, zero_division=0))
if len(np.unique(y_churn_test)) > 1:
    print("Churn ROC-AUC:", round(roc_auc_score(y_churn_test, churn_prob), 4))

scored = test_sessions.copy()
scored["iso_score"] = iso_test_score
scored["anomaly_prob"] = xgb_prob if xgb_prob is not None else (iso_test_score - iso_test_score.min()) / (iso_test_score.max() - iso_test_score.min() + 1e-9)
scored["churn_prob"] = churn_prob
scored["ensembleRiskScore"] = (
    30 * scored["ipChanged"].astype(int)
    + 25 * scored["deviceChanged"].astype(int)
    + 15 * (scored["totalKOs"] >= 3).astype(int)
    + 15 * (scored["maxDownloadsIn2Minutes"] >= 10).astype(int)
    + 15 * (scored["pingPongCount"] >= 2).astype(int)
    + 40 * scored["anomaly_prob"].clip(0, 1)
    + 15 * scored["churn_prob"].clip(0, 1)
).clip(0, 100)
top_risky = scored.sort_values("ensembleRiskScore", ascending=False).head(20)
top_risky.to_csv(ARTIFACTS_DIR / "top_risky_sessions.csv", index=False)
print(top_risky[["sessionId", "primary_anomaly_type", "ensembleRiskScore", "anomaly_prob", "churn_prob", "totalKOs", "ipChanged", "deviceChanged"]])

In [ ]:
daily = events.groupby("eventDate").agg(
    total_events=("action", "size"),
    anomaly_events=("is_anomaly", "sum"),
    download_events=("isDownloadAction", "sum"),
).reset_index().rename(columns={"eventDate": "ds"})

prophet_models = {}
forecast_registry = {}

if Prophet is None:
    print("Prophet not installed; forecasting skipped.")
else:
    try:
        from prophet.serialize import model_to_json
    except Exception:
        model_to_json = None

    for target in ["total_events", "anomaly_events", "download_events"]:
        ts = daily[["ds", target]].rename(columns={target: "y"}).copy()
        if len(ts) < 45:
            print("Skipping", target, "- not enough daily points.")
            continue

        horizon = min(30, max(7, len(ts) // 5))
        train_ts, test_ts = ts.iloc[:-horizon].copy(), ts.iloc[-horizon:].copy()

        eval_model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            seasonality_mode="multiplicative",
        )
        eval_model.fit(train_ts)
        future = eval_model.make_future_dataframe(periods=horizon, freq="D")
        forecast = eval_model.predict(future)
        merged = test_ts.merge(forecast[["ds", "yhat"]], on="ds", how="left")
        mae = mean_absolute_error(merged["y"], merged["yhat"])
        rmse = math.sqrt(mean_squared_error(merged["y"], merged["yhat"]))
        print(f"{target}: Prophet MAE={mae:.3f} RMSE={rmse:.3f}")

        eval_model.plot(forecast, figsize=(10, 5))
        plt.title(f"Prophet - {target}")
        plt.tight_layout()
        plt.show()

        forecast_path = ARTIFACTS_DIR / f"forecast_{target}.csv"
        forecast.to_csv(forecast_path, index=False)

        prod_model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            seasonality_mode="multiplicative",
        )
        prod_model.fit(ts)
        prophet_models[target] = prod_model

        forecast_registry[target] = {
            "forecast_csv": forecast_path.name,
            "mae": float(mae),
            "rmse": float(rmse),
        }

        if model_to_json is not None:
            prophet_json_path = ARTIFACTS_DIR / f"prophet_{target}.json"
            with open(prophet_json_path, "w", encoding="utf-8") as f:
                f.write(model_to_json(prod_model))
            forecast_registry[target]["prophet_json"] = prophet_json_path.name

    with open(ARTIFACTS_DIR / "forecast_registry.json", "w", encoding="utf-8") as f:
        json.dump(forecast_registry, f, ensure_ascii=False, indent=2)

In [ ]:
cluster_mix = (
    sessions["cluster_id"]
    .value_counts(normalize=True)
    .rename_axis("cluster_id")
    .reset_index(name="traffic_share")
    .sort_values("cluster_id")
)
cluster_mix.to_csv(ARTIFACTS_DIR / "cluster_mix.csv", index=False)

dropoff_actions = (
    sessions[sessions["endedAbruptly"] == 1]["lastAction"]
    .value_counts()
    .rename_axis("lastAction")
    .reset_index(name="abrupt_session_count")
)
dropoff_actions.to_csv(ARTIFACTS_DIR / "dropoff_actions.csv", index=False)

path_summary = (
    rare_paths.groupby(["current_action", "actual_next_action"], as_index=False)
    .size()
    .rename(columns={"size": "frequency"})
    .sort_values("frequency", ascending=False)
)
path_summary.to_csv(ARTIFACTS_DIR / "path_summary.csv", index=False)

alerts_feed = scored.sort_values("ensembleRiskScore", ascending=False)[
    [
        "sessionId",
        "insuredId",
        "primary_anomaly_type",
        "ensembleRiskScore",
        "anomaly_prob",
        "churn_prob",
        "totalKOs",
        "ipChanged",
        "deviceChanged",
        "maxDownloadsIn2Minutes",
        "pingPongCount",
    ]
].copy()
alerts_feed.to_csv(ARTIFACTS_DIR / "alerts_feed.csv", index=False)

print("Dashboard-ready exports:")
print(" - cluster_mix.csv")
print(" - dropoff_actions.csv")
print(" - path_summary.csv")
print(" - alerts_feed.csv")

print("\nTop alerts:")
print(alerts_feed.head(10))

In [ ]:
def prepare_like_reference(frame: pd.DataFrame, numeric_cols, categorical_cols, medians, feature_columns):
    work = frame.copy()
    for col in numeric_cols:
        work[col] = pd.to_numeric(work[col], errors="coerce").fillna(medians.get(col, 0.0))
    for col in categorical_cols:
        work[col] = work[col].fillna("UNKNOWN").astype(str)
    numeric_part = work[numeric_cols].astype(float).reset_index(drop=True)
    cat_part = pd.get_dummies(work[categorical_cols].reset_index(drop=True))
    matrix = pd.concat([numeric_part, cat_part], axis=1)
    return matrix.reindex(columns=feature_columns, fill_value=0)


def score_external_dataset(event_csv_path: str, session_csv_path: str | None = None, out_csv_path: str | None = None):
    raw_events = maybe_read_csv(event_csv_path)
    if raw_events is None:
        raise FileNotFoundError(f"Could not find {event_csv_path}")

    ext_events = normalize_events(raw_events)
    ext_sessions = merge_sessions(maybe_read_csv(session_csv_path) if session_csv_path else None, derive_sessions(ext_events))

    X_ext = prepare_like_reference(ext_sessions, num_cols, cat_cols, num_medians, X_train.columns.tolist())
    ext_iso_score = -iso.score_samples(X_ext)
    ext_iso_pred = (ext_iso_score >= iso_threshold).astype(int)

    if xgb is not None:
        ext_anomaly_prob = xgb.predict_proba(X_ext)[:, 1]
    else:
        ext_anomaly_prob = (ext_iso_score - ext_iso_score.min()) / (ext_iso_score.max() - ext_iso_score.min() + 1e-9)

    X_ext_churn = prepare_like_reference(ext_sessions, churn_num, churn_cat, churn_medians, X_churn_train.columns.tolist())
    ext_churn_prob = churn_model.predict_proba(X_ext_churn)[:, 1]

    cluster_input = ext_sessions[cluster_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    ext_cluster = kmeans.predict(cluster_scaler.transform(cluster_input))

    ext_sessions = ext_sessions.copy()
    ext_sessions["iso_score"] = ext_iso_score
    ext_sessions["iso_pred"] = ext_iso_pred
    ext_sessions["anomaly_prob"] = ext_anomaly_prob
    ext_sessions["churn_prob"] = ext_churn_prob
    ext_sessions["cluster_id"] = ext_cluster

    if type_model is not None and type_le is not None:
        ext_type_pred = type_model.predict(X_ext)
        ext_type_prob = type_model.predict_proba(X_ext).max(axis=1)
        ext_sessions["predicted_anomaly_type"] = type_le.inverse_transform(ext_type_pred)
        ext_sessions["predicted_anomaly_type_confidence"] = ext_type_prob
    else:
        ext_sessions["predicted_anomaly_type"] = "unavailable"
        ext_sessions["predicted_anomaly_type_confidence"] = 0.0

    ext_sessions["ensembleRiskScore"] = (
        30 * ext_sessions["ipChanged"].astype(int)
        + 25 * ext_sessions["deviceChanged"].astype(int)
        + 15 * (ext_sessions["totalKOs"] >= 3).astype(int)
        + 15 * (ext_sessions["maxDownloadsIn2Minutes"] >= 10).astype(int)
        + 15 * (ext_sessions["pingPongCount"] >= 2).astype(int)
        + 40 * ext_sessions["anomaly_prob"].clip(0, 1)
        + 15 * ext_sessions["churn_prob"].clip(0, 1)
    ).clip(0, 100)

    if out_csv_path:
        ext_sessions.to_csv(out_csv_path, index=False)
        print("Saved scored sessions to", out_csv_path)

    return ext_sessions


# Example:
# scored_external = score_external_dataset("anomaly_test_set.csv", out_csv_path="scored_external_sessions.csv")
# scored_external.sort_values("ensembleRiskScore", ascending=False).head(10)

In [ ]:
feature_bundle = {
    "session_numeric_features": num_cols,
    "session_categorical_features": cat_cols,
    "cluster_numeric_features": cluster_cols,
    "iso_threshold": float(iso_threshold),
    "uses_xgboost_binary": bool(xgb is not None),
    "uses_random_forest_type": bool(type_model is not None),
    "uses_prophet": bool(Prophet is not None),
}
with open(ARTIFACTS_DIR / "feature_bundle.json", "w", encoding="utf-8") as f:
    json.dump(feature_bundle, f, ensure_ascii=False, indent=2)

cluster_scaler_payload = {
    "feature_order": cluster_cols,
    "mean": [float(v) for v in cluster_scaler.mean_],
    "scale": [float(v) for v in cluster_scaler.scale_],
}
with open(ARTIFACTS_DIR / "cluster_scaler_params.json", "w", encoding="utf-8") as f:
    json.dump(cluster_scaler_payload, f, ensure_ascii=False, indent=2)

try:
    from skl2onnx import to_onnx
    from skl2onnx.common.data_types import FloatTensorType
    from sklearn.pipeline import Pipeline
except Exception as exc:
    print("skl2onnx not available; ONNX export for sklearn models skipped:", exc)
else:
    session_input = [("input", FloatTensorType([None, len(X_train.columns)]))]
    churn_input = [("input", FloatTensorType([None, len(X_churn_train.columns)]))]
    cluster_input = [("input", FloatTensorType([None, len(cluster_cols)]))]

    try:
        onnx_iso = to_onnx(iso, initial_types=session_input, target_opset=13)
        with open(ARTIFACTS_DIR / "iso_binary.onnx", "wb") as f:
            f.write(onnx_iso.SerializeToString())
    except Exception as exc:
        with open(ARTIFACTS_DIR / "iso_binary_fallback.json", "w", encoding="utf-8") as f:
            json.dump(
                {
                    "threshold": float(iso_threshold),
                    "note": "ONNX conversion unavailable for IsolationForest in this runtime. Use xgb_binary.onnx as the primary Spring Boot binary detector.",
                },
                f,
                ensure_ascii=False,
                indent=2,
            )
        print("Isolation Forest ONNX export skipped:", exc)

    if type_model is not None:
        onnx_type = to_onnx(type_model, initial_types=session_input, target_opset=13)
        with open(ARTIFACTS_DIR / "rf_type.onnx", "wb") as f:
            f.write(onnx_type.SerializeToString())

    onnx_churn = to_onnx(churn_model, initial_types=churn_input, target_opset=13)
    with open(ARTIFACTS_DIR / "rf_churn.onnx", "wb") as f:
        f.write(onnx_churn.SerializeToString())

    cluster_pipeline = Pipeline([("scaler", cluster_scaler), ("kmeans", kmeans)])
    onnx_cluster = to_onnx(cluster_pipeline, initial_types=cluster_input, target_opset=13)
    with open(ARTIFACTS_DIR / "kmeans_persona_pipeline.onnx", "wb") as f:
        f.write(onnx_cluster.SerializeToString())

try:
    import onnxmltools
    from skl2onnx.common.data_types import FloatTensorType
except Exception as exc:
    print("onnxmltools not available; XGBoost ONNX export skipped:", exc)
else:
    if xgb is not None:
        xgb_onnx = onnxmltools.convert_xgboost(
            xgb,
            initial_types=[("input", FloatTensorType([None, len(X_train.columns)]))],
            target_opset=13,
        )
        onnxmltools.utils.save_model(xgb_onnx, ARTIFACTS_DIR / "xgb_binary.onnx")

In [ ]:
import zipfile

joblib.dump(iso, ARTIFACTS_DIR / "iso_binary.joblib")
if xgb is not None:
    joblib.dump(xgb, ARTIFACTS_DIR / "xgb_binary.joblib")
if type_model is not None:
    joblib.dump(type_model, ARTIFACTS_DIR / "rf_type.joblib")
    with open(ARTIFACTS_DIR / "rf_type_labels.json", "w", encoding="utf-8") as f:
        json.dump({int(i): label for i, label in enumerate(type_le.classes_)}, f, ensure_ascii=False, indent=2)
joblib.dump(churn_model, ARTIFACTS_DIR / "rf_churn.joblib")
joblib.dump(kmeans, ARTIFACTS_DIR / "kmeans_persona.joblib")
joblib.dump(cluster_scaler, ARTIFACTS_DIR / "cluster_scaler.joblib")

with open(ARTIFACTS_DIR / "session_feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(X_train.columns.tolist(), f, ensure_ascii=False, indent=2)
with open(ARTIFACTS_DIR / "session_numeric_medians.json", "w", encoding="utf-8") as f:
    json.dump(num_medians, f, ensure_ascii=False, indent=2)
with open(ARTIFACTS_DIR / "binary_feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(X_train.columns.tolist(), f, ensure_ascii=False, indent=2)
with open(ARTIFACTS_DIR / "type_feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(X_train.columns.tolist(), f, ensure_ascii=False, indent=2)
with open(ARTIFACTS_DIR / "churn_feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(X_churn_train.columns.tolist(), f, ensure_ascii=False, indent=2)
with open(ARTIFACTS_DIR / "churn_numeric_medians.json", "w", encoding="utf-8") as f:
    json.dump(churn_medians, f, ensure_ascii=False, indent=2)
with open(ARTIFACTS_DIR / "markov_transition_lookup.json", "w", encoding="utf-8") as f:
    json.dump({src: grp[["to_action", "probability"]].to_dict("records") for src, grp in lookup.items()}, f, ensure_ascii=False, indent=2)

preferred_binary_model = "xgb_binary.onnx" if (ARTIFACTS_DIR / "xgb_binary.onnx").exists() else ("iso_binary.onnx" if (ARTIFACTS_DIR / "iso_binary.onnx").exists() else "iso_binary.joblib")
deployment_manifest = {
    "binary_detection": {
        "preferred": preferred_binary_model,
        "fallback": "iso_binary.joblib",
        "feature_columns": "binary_feature_columns.json",
        "numeric_medians": "session_numeric_medians.json",
    },
    "anomaly_type": {
        "model": "rf_type.onnx" if (ARTIFACTS_DIR / "rf_type.onnx").exists() else "rf_type.joblib",
        "feature_columns": "type_feature_columns.json",
        "labels": "rf_type_labels.json" if type_model is not None else None,
    },
    "churn": {
        "model": "rf_churn.onnx" if (ARTIFACTS_DIR / "rf_churn.onnx").exists() else "rf_churn.joblib",
        "feature_columns": "churn_feature_columns.json",
        "numeric_medians": "churn_numeric_medians.json",
    },
    "clustering": {
        "model": "kmeans_persona_pipeline.onnx" if (ARTIFACTS_DIR / "kmeans_persona_pipeline.onnx").exists() else "kmeans_persona.joblib",
        "cluster_features": cluster_cols,
        "scaler_params": "cluster_scaler_params.json",
    },
    "next_action": {
        "artifact": "markov_transition_lookup.json",
    },
    "forecasting": forecast_registry if "forecast_registry" in globals() else {},
    "dashboard_exports": [
        "alerts_feed.csv",
        "cluster_mix.csv",
        "dropoff_actions.csv",
        "path_summary.csv",
        "top_risky_sessions.csv",
    ],
}
with open(ARTIFACTS_DIR / "deployment_manifest.json", "w", encoding="utf-8") as f:
    json.dump(deployment_manifest, f, ensure_ascii=False, indent=2)

bundle_path = Path("springboot_bundle_tabular.zip")
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for artifact in sorted(ARTIFACTS_DIR.iterdir()):
        if artifact.is_file():
            zf.write(artifact, arcname=artifact.name)

LLM_PROMPT_TEMPLATE = """
You are an AIOps and security analyst.

SESSION:
  session_id: {session_id}
  insured_id: {insured_id}
  persona_cluster: {cluster_id}
  country: {country}
  total_events: {total_events}
  duration_seconds: {duration_seconds}

MODEL OUTPUTS:
  isolation_forest_score: {iso_score:.4f}
  anomaly_probability: {anomaly_prob:.4f}
  anomaly_type: {anomaly_type}
  churn_probability: {churn_prob:.4f}
  ensemble_risk_score: {risk_score:.1f}
  top_markov_next_actions: {next_actions}
  forecast_signal: {forecast_signal}

TASK:
1. Explain why the session is normal, suspicious, or critical.
2. Identify the strongest behavior signals.
3. Suggest one concrete action for the dashboard team.
"""

print("Artifacts saved in", ARTIFACTS_DIR.resolve())
print("Bundle created:", bundle_path.resolve())
for artifact in sorted(p.name for p in ARTIFACTS_DIR.iterdir()):
    print(" -", artifact)